# Ejercicio 8: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [7]:
import kagglehub
from aiohttp import payload
from kagglehub import KaggleDatasetAdapter

In [8]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'wikipedia-text-corpus-for-nlp-and-llm-projects' dataset.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [9]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [10]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [11]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [10]:
embeddings = np.array(embeddings)

np.save("embeddings.npy", embeddings)

In [13]:
embeddings = np.load("embeddings.npy")

In [15]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [13]:
! pip install faiss-gpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.5 MB/s eta 0:00:00:00:0100:01


In [14]:
import faiss
import numpy as np

# Embeddings normalizados: usar producto interno para similitud coseno

# Convertir embeddings a float32 y almacenarlos de forma contigua
xb = np.ascontiguousarray(embeddings, dtype="float32")
xq = np.ascontiguousarray(query_vec, dtype="float32")

# Dimensión de los embeddings
d = xb.shape[1]

index = faiss.IndexFlatIP(d)

# Agregar los embeddings al índice
index.add(xb)

# Buscar los 10 vecinos más cercanos
scores, indices = index.search(xq, k=10)

# Mientras más pequeña la distancia, más similares son los vectores.
scores

array([[0.87034863, 0.86180043, 0.8401017 , 0.83913344, 0.8385889 ,
        0.8345113 , 0.8343499 , 0.8316221 , 0.83021617, 0.82664007]],
      dtype=float32)

In [15]:
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"{rank:2d}. idx={idx:6d} score={score:.4f} text={chunks_df.iloc[idx]['text'][:120]}")

 1. idx= 10176 score=0.8703 text=Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro
 2. idx=     1 score=0.8618 text=Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter
 3. idx= 10177 score=0.8401 text=ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries
 4. idx= 37406 score=0.8391 text=ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply 
 5. idx= 71872 score=0.8386 text=is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa
 6. idx= 37409 score=0.8345 text=shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement
 7. idx= 10481 score=0.8343 text=Current sense monitor A Current Sense Monit

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [16]:
! pip install qdrant-client --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 29.3 MB/s eta 0:00:00


In [17]:
print(chunks_df.columns)

Index(['doc_id', 'chunk_id', 'text'], dtype='object')


In [18]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import numpy as np

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="collection",
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)
points = []
for i, row in chunks_df.iterrows():
    points.append(PointStruct(
        id=int(i),
        vector=embeddings[i],
        payload={
            "doc_id": row["doc_id"],
            "chunk_id": row["chunk_id"],
            "text": row["text"],
        }
    ))
client.upsert(collection_name="collection", wait=True, points=points)


/tmp/ipykernel_832/3567462416.py:22: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 79104 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name="collection", wait=True, points=points)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [19]:
client.get_collection("collection")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=79104, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimiz

In [20]:
# [m for m in dir(client) if "query" in m.lower() or "search" in m.lower()]

In [21]:
# help(client.query_points)

In [22]:
def qdrant_search(query_embedding, k):
    search_result = client.query_points(
        collection_name="collection",
        query= query_embedding[0],
        with_payload=True,
        limit=k
    ).points

    formatted_result = []
    for point in search_result:
        formatted_result.append((point.id, point.score, point.payload["text"], point.payload))
    return formatted_result

In [23]:
resultados = qdrant_search(query_vec, 5)
for r in resultados:
    print(f"{r}\n\n")

(10176, 0.8703485972057368, "Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or its voltage output, to a more comprehensive testing of the battery's condition, namely its capacity for accumulating charge and any possible flaws affecting the battery's performance and security. The most simple battery tester is a DC ammeter, that indicates the battery's charge rate. DC voltmeters can be used to estimate the charge rate of a battery, provided that its nominal voltage is known. There are many types of integrated battery testers, each one corresponding to a specific condition testing procedure, according to the type of battery being tested, such as the â€œ421â€\x9d test for lead-ac", {'doc_id': 1391, 'chunk_id': 0, 'text': "Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a 

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [21]:
pip install "pymilvus==2.5.8" "milvus-lite==2.5.1" --quiet

  Using cached pymilvus-2.5.8-py3-none-any.whl.metadata (5.7 kB)
Using cached pymilvus-2.5.8-py3-none-any.whl (227 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 22.0 MB/s eta 0:00:00:00:0100:01


In [31]:
from pymilvus import MilvusClient, DataType

client = MilvusClient("milvus_vectorialDB.db")

if client.has_collection("wiki_collection"):
    client.drop_collection("wiki_collection")


schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=False
)


# Llave primaria
schema.add_field(
    field_name="id",
    datatype=DataType.INT64,
    is_primary=True
)


# Embedding
schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=embeddings.shape[1]
)


# Metadata
schema.add_field(
    field_name="doc_id",
    datatype=DataType.INT64
)


schema.add_field(
    field_name="chunk_id",
    datatype=DataType.INT64
)


schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=8192
)


client.create_collection(
    collection_name="wiki_collection",
    schema=schema
)

In [34]:
print(client.list_collections())

['wiki_collection']


In [35]:
client.describe_collection("wiki_collection")

{'collection_name': 'wiki_collection',
 'auto_id': False,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 0,
   'name': 'id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {},
   'is_primary': True},
  {'field_id': 0,
   'name': 'embedding',
   'description': '',
   'type': <DataType.FLOAT_VECTOR: 101>,
   'params': {'dim': 768}},
  {'field_id': 0,
   'name': 'doc_id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {}},
  {'field_id': 0,
   'name': 'chunk_id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {}},
  {'field_id': 0,
   'name': 'text',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 8192}}],
 'functions': [],
 'aliases': [],
 'collection_id': 0,
 'consistency_level': 0,
 'properties': {},
 'num_partitions': 1,
 'enable_dynamic_field': False}

In [39]:
data = []

for i, row in chunks_df.iterrows():

    data.append({
        "id": i,
        "embedding": embeddings[i].tolist(),
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"]),
        "text": row["text"]
    })

In [42]:
def insert_in_chunks(client, collection, data, chunk_size=1000):
    if not isinstance(data, list):
        raise TypeError("data must be a list of dictionaries")

    if not data:
        return

    if not all(isinstance(row, dict) for row in data):
        raise TypeError("data must be a list of dictionaries, where each dictionary is one record")

    n = len(data)

    for i in range(0, n, chunk_size):
        chunk = data[i:i + chunk_size]
        client.insert(collection_name=collection, data=chunk)


In [43]:
insert_in_chunks(client, "wiki_collection", data)

In [44]:
print(
    client.get_collection_stats(
        collection_name="wiki_collection"
    )
)

{'row_count': 79104}


In [45]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="COSINE",
    params={
        "M": 16,
        "efConstruction": 200
    }
)

client.create_index(
    collection_name="wiki_collection",
    index_params=index_params
)

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 610, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 996, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


In [50]:
def milvus_search(query_embedding, k):
    return client.search(
        collection_name="wiki_collection",
        data=query_embedding,
        limit=k
    )

In [51]:
milvus_search(query_vec, 5)

ERROR:milvus_lite.adapter.grpc.servicer:Search failed: function_score
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/milvus_lite/adapter/grpc/servicer.py", line 441, in Search
  File "/usr/local/lib/python3.12/dist-packages/milvus_lite/adapter/grpc/translators/search.py", line 124, in parse_search_request
AttributeError: function_score
2026-07-10 05:10:42,189 [ERROR][handler]: RPC error: [search], <MilvusException: (code=1, message=function_score)>, <Time:{'RPC start': '2026-07-10 05:10:42.182358', 'RPC error': '2026-07-10 05:10:42.189353'}> (decorators.py:140)
2026-07-10 05:10:42,189 [ERROR][search]: Failed to search collection: wiki_collection (milvus_client.py:411)


MilvusException: <MilvusException: (code=1, message=function_score)>

In [52]:
milvus_search(query_vec, 20)

ERROR:milvus_lite.adapter.grpc.servicer:Search failed: function_score
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/milvus_lite/adapter/grpc/servicer.py", line 441, in Search
  File "/usr/local/lib/python3.12/dist-packages/milvus_lite/adapter/grpc/translators/search.py", line 124, in parse_search_request
AttributeError: function_score
2026-07-10 05:11:07,431 [ERROR][handler]: RPC error: [search], <MilvusException: (code=1, message=function_score)>, <Time:{'RPC start': '2026-07-10 05:11:07.424631', 'RPC error': '2026-07-10 05:11:07.431491'}> (decorators.py:140)
2026-07-10 05:11:07,432 [ERROR][search]: Failed to search collection: wiki_collection (milvus_client.py:411)


MilvusException: <MilvusException: (code=1, message=function_score)>

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?
